## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [3]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil

# Розрахунок effect size для пропорцій
effect_size = sms.proportion_effectsize(0.19, 0.2)

# Скільки треба спостережень у кожній групі
required_n = sms.NormalIndPower().solve_power(
    effect_size,
    power=0.8,
    alpha=0.05,
    ratio=1
)

required_n = ceil(required_n) 

print(f"Необхідна кількість користувачів на групу: {required_n}")
print(f"Сумарно для двох груп: {required_n*2}")


Необхідна кількість користувачів на групу: 24638
Сумарно для двох груп: 49276


2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [4]:
df = pd.read_csv(r"D:\Навчання\Data Analytics\КУРС DATA ANALIST\python\DATA\cookie_cats.csv")

df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [8]:
df.groupby("version")['retention_7'].mean()

version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64

**Гіпотеза**

H₀: зміна рівня воріт не впливає на утримання через 7 днів.  
H₁: зміщення воріт на рівень 40 зменшує утримання.  

На основі отриманих середніх бачимо, що утримання трохи менше в тестовій групі (18,2% < 19%), але різниця невелика.
Щоб точно сказати, чи це статистично значуще, треба провести додаткові тести.

3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [10]:
# перевіримо чи немає користувачів, які були обрані кілька разів.
session_counts = df['userid'].value_counts(ascending=False)
multi_users = session_counts[session_counts > 1].count()

print(f'Є {multi_users} користувачів, які зустрічаються кілька разів у наборі даних.')

Є 0 користувачів, які зустрічаються кілька разів у наборі даних.


In [11]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

# Визначимо успіхи (кількість користувачів, які повернулися) та розміри груп
control = df[df['version'] == 'gate_30']
treatment = df[df['version'] == 'gate_40']

count = [control['retention_7'].sum(), treatment['retention_7'].sum()]  # кількість "успіхів"
nobs = [len(control), len(treatment)]  # розмір груп

# z-тест для двох пропорцій (двосторонній тест)
z_stat, p_value = proportions_ztest(count, nobs)

# 95% довірчі інтервали для кожної групи
confint_control = proportion_confint(count[0], nobs[0], alpha=0.05, method='normal')
confint_treatment = proportion_confint(count[1], nobs[1], alpha=0.05, method='normal')

# Вивід результатів
print(f"z statistic: {z_stat:.3f}")
print(f"p-value: {p_value:.4f}")
print(f"Довірчий інтервал 95% для групи control: [{confint_control[0]:.3f}, {confint_control[1]:.3f}]")
print(f"Довірчий інтервал 95% для групи treatment: [{confint_treatment[0]:.3f}, {confint_treatment[1]:.3f}]")


z statistic: 3.164
p-value: 0.0016
Довірчий інтервал 95% для групи control: [0.187, 0.194]
Довірчий інтервал 95% для групи treatment: [0.178, 0.186]


**Висновки**

p-value < 0.05, отже різниця між групами статистично значуща.
Це означає, що зміщення воріт на рівень 40 реально вплинуло на утримання користувачів.

Довірчі інтервали не перетинаються, тобто ми можемо бути досить впевнені, що утримання в контрольній групі (ворота на рівні 30) вище, ніж у тестовій (ворота на рівні 40).

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


**Гіпотези для тесту Хі-квадрат**

H₀: версія гри не впливає на утримання користувачів → дві змінні незалежні.  
H₁: версія гри впливає на утримання → є залежність між версією і retention_7.

In [16]:
# Формуємо таблицю спряженості
table = pd.crosstab(df['version'], df['retention_7'])

# Виконуємо тест Хі-квадрат
chi2_stat, p_value, dof, expected = chi2_contingency(table)

# Вивід результатів
print(f"χ² = {chi2_stat:.3f}")
print(f"Ступені свободи = {dof}")
print(f"p-value = {p_value:.4f}")
print("Очікувані частоти:\n", expected)


χ² = 9.959
Ступені свободи = 1
p-value = 0.0016
Очікувані частоти:
 [[36382.90257127  8317.09742873]
 [37025.09742873  8463.90257127]]


**Висновки**  
Є статистично значуща залежність між версією гри і утриманням на 7-й день (p = 0.0016 < 0.05).  
Це підтверджує, що зміщення воріт на рівень 40 вплинуло на поведінку користувачів.

Очікувані частоти показують, що спостережувані дані відрізняються від очікуваних, що і призводить до великої χ²-статистики.
